The goal for this file is twofold: 
1) To tag each municipality as 1st to 5th income class as provided by the DOF
2) To the connect the Immunization data from 2018 to 2026 as provided by the FSIS.

In [1]:
import pandas as pd
import numpy as np
from functools import reduce
import os 

pd.set_option('display.max_columns', None)


# Tagging the Income Classification

With the Excel file provided about the PSGC codes, the tags on the income class are not actually provided. [DeepSeek](https://chat.deepseek.com/share/34sb7ym87whiiwjsj3) was used to convert [DOF PDF](https://blgf.gov.ph/wp-content/uploads/2025/01/DOF-DO-074.2024.pdf) into an Excel. Further cleaning will be done because it only has the names of the municipalities, and not the actual codes.

In [2]:
income_df = pd.read_excel('data/income/income.xlsx').drop_duplicates()
income_df

,REGION,Province Name,Geographic Level,LGU NAME,OLD CLASS,NEW CLASS
0,CAR,Abra,Province,Abra,3rd,1st
1,CAR,Apayao,Province,Apayao,3rd,2nd
2,CAR,Benguet,Province,Benguet,2nd,1st
3,CAR,Ifugao,Province,Ifugao,3rd,2nd
4,CAR,Kalinga,Province,Kalinga,3rd,2nd
...,...,...,...,...,...,...
1824,BARMM,Tawi-Tawi,Mun,Simunul,4th,2nd
1825,BARMM,Tawi-Tawi,Mun,Sitangkai,1st,1st
1826,BARMM,Tawi-Tawi,Mun,South Ubian,3rd,2nd
1827,BARMM,Tawi-Tawi,Mun,Tandubas,2nd,1st


# Adding the KPIs from the FHSIS

From a first look through file system and the files within it, we know that:
* The granularity of the data vary based on the years. We have that:
    * For 2018 to 2024 data, it is annual, and we only have at most provincial- and city- level data.
    * For 2025 and 2026 data, the granularity is monthly and municipal-level. However, the 2026 data is only until February.
* The key performance indicators that the FHSIS reports are:
    * Fully Immunized Children (FIC) - The monthly target is 7.92%, which comes from the annual target of 95% divided by the 12 months.
        * According to the [DOH](https://doh.gov.ph/wp-content/uploads/2023/08/Booklet-7-Monitoring-Supportive-Supervision-and-Evaluation.pdf), for a child to be fully immunized, they need to have the following in the list. The FSIS also report the vaccination rates below:
            * One (1) dose of bCG at birth or anytime, 
            * Three (3) doses of OPV, 
            * Three (3) doses of Pentavalent vaccines; and 
            * One (1) dose of Measles-containing vaccine (MCV)
                * For this one, the FSIS records this as Two (2) doses of Measles Mumps Rubella (2 MMR) vaccine
    * Completely Immunized Children (CIC) - This one does not seem as important

Overall plan:
* Because of the differing granularities of the dataset, we will have two separate dataframes. They are:
    * Annual and provincial-level data from 2018 to 2025
    * Monthly and municipal-level data from 2025 to February 2026

For the naming schemes they are:

1. Annual Immunization Dataset – Column Definitions

| Column Name  | Description                                          |
| ------------ | ---------------------------------------------------- |
| Area         | Geographic area (e.g., region, province, or city)    |
| Eligible_Pop | Total number of individuals eligible for vaccination |

2. At Birth Indicators

| Column Name | Description                                                                                  |
| ----------- | -------------------------------------------------------------------------------------------- |
| CPAB        | Children Protected at Birth; infants protected against tetanus through maternal immunization |
| BCG         | Bacillus Calmette–Guérin vaccine; protects against tuberculosis                              |
| HEPA_B1     | Hepatitis B Birth Dose; first dose given at birth                                            |

3. Primary Series Vaccines

| Column Name | Description                                                             |
| ----------- | ----------------------------------------------------------------------- |
| DPT_1       | First dose of DPT-containing vaccine (Diphtheria, Pertussis, Tetanus)   |
| DPT_2       | Second dose of DPT-containing vaccine                                   |
| DPT_3       | Third dose of DPT-containing vaccine (completion of primary DPT series) |
| OPV_1       | First dose of Oral Polio Vaccine                                        |
| OPV_2       | Second dose of Oral Polio Vaccine                                       |
| OPV_3       | Third dose of Oral Polio Vaccine                                        |
| IPV         | Inactivated Polio Vaccine                                               |

4. Pneumococcal Vaccines

| Column Name | Description                                   |
| ----------- | --------------------------------------------- |
| PCV_1       | First dose of Pneumococcal Conjugate Vaccine  |
| PCV_2       | Second dose of Pneumococcal Conjugate Vaccine |
| PCV_3       | Third dose of Pneumococcal Conjugate Vaccine  |

5. Measles-Containing Vaccines

| Column Name | Description                               |
| ----------- | ----------------------------------------- |
| MCV_1       | First dose of Measles-Containing Vaccine  |
| MCV_2       | Second dose of Measles-Containing Vaccine |

6. Immunization Coverage Indicators

| Column Name | Description                                                                                                      |
| ----------- | ---------------------------------------------------------------------------------------------------------------- |
| FIC         | Fully Immunized Child; child who has received all recommended basic vaccines                                     |
| CIC         | Completely Immunized Child; broader definition depending on program (may include additional vaccines beyond FIC) |

---

Each vaccination indicator is further disaggregated and represented using the following suffix-based naming convention:

* `_M` for Male recipients
* `_F` for Female recipients
* `_Total` for Total recipients (Male + Female)
* `_Percent` for Proportion of vaccinated individuals relative to the eligible population

This means that for each base indicator (e.g., `DPT_1`, `BCG`, `FIC`), the dataset contains four corresponding columns:

* `{Indicator}_M`
* `{Indicator}_F`
* `{Indicator}_Total`
* `{Indicator}_Percent`

### Cleaning 2018 Data

Based on the table summary, we need the following tables:
* Table 2D.1	Proportion of FIC, CIC and Children Protected at Birth
* Table 2D.2	Proportion of Children given BCG and Hepatitis B1 Vaccines
* Table 2D.3	Proportion of Children given Pentavalent Vaccines
* Table 2D.4	Proportion of Children given Oral Polio Vaccines (OPV)
* Table 2D.5	Proportion of Children given Measles-containing Vaccines (MCV) and Rotavirus Vaccines


In [3]:
TEMP_COLUMN_NAMES = [
 'Area',
 'CPAB',
 'BCG',
 'HEPA_B1',
 'DPT_1',
 'DPT_2',
 'DPT_3',
 'OPV_1',
 'OPV_2',
 'OPV_3',
 'IPV_1',
 'IPV_2',
 'PCV_1',
 'PCV_2',
 'PCV_3',
 'MCV_1',
 'MCV_2',
 'FIC',
 'CIC']

CODE_COLUMN_NAMES = TEMP_COLUMN_NAMES[:1]
for indicator in TEMP_COLUMN_NAMES[1:]:
    CODE_COLUMN_NAMES.append(indicator + '_M')
    CODE_COLUMN_NAMES.append(indicator + '_F')
    CODE_COLUMN_NAMES.append(indicator + '_Total')
    CODE_COLUMN_NAMES.append(indicator + '_Percent')

def create_column_names(original_column_names, is_PSGC=False):
    """Add the _M, _F, _Total, _Percent to all indicators"""
    code_column_names = ['Area', 'Eligible_Pop']
    if is_PSGC:
        code_column_names = ['PSGC', 'Area', 'Eligible_Pop']

    for indicator in original_column_names:
        code_column_names.append(indicator + '_M')
        code_column_names.append(indicator + '_F')
        code_column_names.append(indicator + '_Total')
        code_column_names.append(indicator + '_Percent')

    return code_column_names

YEAR_CODE_COLUMN_NAMES = CODE_COLUMN_NAMES.copy()
YEAR_CODE_COLUMN_NAMES.insert(1, 'Year')

In [4]:
# 2018 data

sheet_filepath = 'data/medical/2018-2024/CC 2018.xlsx'
# TABLE 1
table_1_2018_df = pd.read_excel(sheet_filepath, 
                                   sheet_name='Table 2D.1',
                                   skiprows=7,
                                   skipfooter=3)

# get rid of the 10th column (number of live births) and the last column
table_1_2018_df = table_1_2018_df.drop(table_1_2018_df.columns[10], axis=1).copy()
table_1_2018_df = table_1_2018_df.drop(table_1_2018_df.columns[-1], axis=1).copy()

# rename columns
table_1_2018_df.columns = create_column_names(['FIC', 'CIC', 'CPAB'])
table_1_2018_df.dropna(inplace=True)

# TABLE 2
table_2_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.2',
                                skiprows=7, skipfooter=3)

# drop the last three columns
table_2_2018_df = table_2_2018_df.drop(table_2_2018_df.columns[-4:], axis=1).copy()
table_2_2018_df.columns = create_column_names(['BCG', 'HEPA_B1'])
table_2_2018_df.dropna(inplace=True)

# TABLE 3
table_3_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.3',
                                skiprows=7, skipfooter=3)

table_3_2018_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'])
table_3_2018_df.dropna(inplace=True)

# TABLE 4 
table_4_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.4',
                                skiprows=7, skipfooter=3)

table_4_2018_df.columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3'])
ipv_cols = create_column_names(['IPV_1', 'IPV_2'])[2:]
table_4_2018_df[ipv_cols] = 0
table_4_2018_df.dropna(inplace=True)

# TABLE 5
table_5_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.5',
                                skiprows=7, skipfooter=3)
table_5_2018_df = table_5_2018_df.drop(table_5_2018_df.columns[-8:], axis=1).copy()
table_5_2018_df.columns = create_column_names(['MCV_1', 'MCV_2'])
table_5_2018_df.dropna(inplace=True)

# TABLE 6
table_6_2018_df = pd.read_excel(sheet_filepath, sheet_name='Table 2D.6',
                                skiprows=7,skipfooter=3)
table_6_2018_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'])
table_6_2018_df['Area'] = table_6_2018_df['Area'].replace('N C R 1', 'N C R')
table_6_2018_df.dropna(inplace=True)

table_dfs = [
    table_1_2018_df,
    table_2_2018_df,
    table_3_2018_df,
    table_4_2018_df,
    table_5_2018_df,
    table_6_2018_df
]

# remove Eligible_Pop column
for table_df in table_dfs:
    table_df.drop(columns='Eligible_Pop', inplace=True)

table_2018_df = reduce(
    lambda left, right: pd.merge(
        left, right,
        on=['Area'],
        how='inner'
    ),
    table_dfs
)

table_2018_df['Year'] = 2018

table_2018_df = table_2018_df[YEAR_CODE_COLUMN_NAMES].copy()

### Cleaning 2019 data

In [5]:
table_2019_df = pd.read_excel('data/medical/2018-2024/CC 2019.xlsx', 
                              sheet_name='Immunization',
                              skiprows=5, skipfooter=3).dropna()
table_2019_df.drop(columns=table_2019_df.columns[1], inplace=True)

# deal with the new IPV_2, so find where IPV_1 ends
insert_at = table_2019_df.columns.get_loc('%.9') + 1

# new IPV_2 columns
new_cols = [
    'IPV_2_M', 'IPV_2_F', 'IPV_2_Total', 'IPV_2_Percent'
]

# insert them in order
for i, col in enumerate(new_cols):
    table_2019_df.insert(insert_at + i, col, 0)   

table_2019_df.columns = CODE_COLUMN_NAMES
table_2019_df['Year'] = 2019
table_2019_df = table_2019_df[YEAR_CODE_COLUMN_NAMES].copy()

#### Cleaning 2020 & 2021 data

The files for 2020 to 2022 have a semi-consistent format, so I will just use a function to extract the information. If there are any if statements, they are mainly for the weird files.

In [6]:
def create_table_df_version1(sheet_filepath, year):
    # Table 1
    table1_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=0)
    table1_df = table1_df.iloc[:, :14].copy()
    table1_df.columns = create_column_names(['CPAB', 'BCG', 'HEPA_B1'])
    table1_df.dropna(inplace=True)

    # Table 2
    table2_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=1)
    table2_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'])
    table2_df.dropna(inplace=True)

    # Table 3
    table3_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=2)
    opv_ipv_columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3', 'IPV_1'])
    table3_df.columns = opv_ipv_columns
    ipv2_columns = create_column_names(['IPV_2'])[2:]
    table3_df[ipv2_columns] = 0
    table3_df.dropna(inplace=True)

    # Table 4
    table4_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=3)
    table4_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'])
    table4_df.dropna(inplace=True)

    # Table 5
    table5_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=4)
    mcv_only_cols = create_column_names(['MCV_1', 'MCV_2'])
    if len(table5_df.columns) != len(mcv_only_cols):
        table5_df.drop(columns=table5_df.columns[6], inplace=True)
    table5_df.columns = mcv_only_cols
    table5_df.dropna(inplace=True)

    # Table 6
    table6_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=4)
    fic_cic_only_cols = create_column_names(['FIC', 'CIC'])
    if len(table6_df.columns) != len(fic_cic_only_cols):
        table6_df.drop(columns=table6_df.columns[6], inplace=True)
    table6_df.columns = fic_cic_only_cols
    table6_df.dropna(inplace=True)

    table_dfs = [
        table1_df,
        table2_df,
        table3_df,
        table4_df,
        table5_df,
        table6_df
    ]

    for table_df in table_dfs:
        table_df.drop(columns='Eligible_Pop', inplace=True)

    table_df = reduce(
        lambda left, right: pd.merge(
            left, right,
            on=['Area'],
            how='inner'
        ),
        table_dfs
    )

    table_df['Year'] = year

    table_df = table_df[YEAR_CODE_COLUMN_NAMES]

    return table_df

In [7]:
table_2020_df = create_table_df_version1('data/medical/2018-2024/CC 2020.xlsx', 2020)

table_2021_df = create_table_df_version1('data/medical/2018-2024/CC 2021.xlsx', 2021)

### Cleaning 2022 to 2024 data

In [8]:
def create_table_df_version2(sheet_filepath, year, is_psgc):
    # Table 1
    table1_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=0)
    #table1_df = table1_df.iloc[:, :14].copy()
    table1_df.columns = create_column_names(['CPAB', 'BCG', 'HEPA_B1'], is_PSGC=is_psgc)
    table1_df.dropna(inplace=True)

    # Table 2
    table2_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=1)
    table2_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'], is_PSGC=is_psgc)
    table2_df.dropna(inplace=True)

    # # Table 3
    table3_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=2)
    opv_columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3'], is_PSGC=is_psgc)
    table3_df.columns = opv_columns
    table3_df.dropna(inplace=True)

    # Table 3.5 (IPV)
    table_ipv_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=3)
    if is_psgc:
        table_ipv_df = table_ipv_df.iloc[:, :11]
    else:
        table_ipv_df = table_ipv_df.iloc[:, :10]
    table_ipv_df.columns = create_column_names(['IPV_1', 'IPV_2'], is_PSGC=is_psgc)
    table_ipv_df.dropna(inplace=True)

    # Table 4
    table4_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=4)
    table4_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'], is_PSGC=is_psgc)
    table4_df.dropna(inplace=True)

    # Table 5
    table5_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=5)
    mcv_only_cols = create_column_names(['MCV_1', 'MCV_2'], is_PSGC=is_psgc)
    if len(table5_df.columns) != len(mcv_only_cols):
        table5_df.drop(columns=table5_df.columns[6], inplace=True)
    table5_df.columns = mcv_only_cols
    table5_df.dropna(inplace=True)

    # Table 6
    table6_df = pd.read_excel(sheet_filepath, skiprows=5, sheet_name=6)
    fic_cic_only_cols = create_column_names(['FIC', 'CIC'], is_PSGC=is_psgc)
    if len(table6_df.columns) != len(fic_cic_only_cols):
        table6_df.drop(columns=table6_df.columns[6], inplace=True)
    table6_df.columns = fic_cic_only_cols
    table6_df.dropna(inplace=True)

    table_dfs = [
        table1_df,
        table2_df,
        table3_df,
        table_ipv_df,
        table4_df,
        table5_df,
        table6_df
    ]

    for table_df in table_dfs:
        table_df.drop(columns='Eligible_Pop', inplace=True)

    table_df['Year'] = year

    if is_psgc:
        table_df = reduce(
            lambda left, right: pd.merge(
                left, right,
                on=['Area', 'PSGC'],
                how='inner'
            ),
            table_dfs
        )

        PSGC_CODE_COLUMN_NAMES = YEAR_CODE_COLUMN_NAMES[:1] + ['PSGC'] + YEAR_CODE_COLUMN_NAMES[1:]
        return table_df[PSGC_CODE_COLUMN_NAMES].copy()

    else:
        table_df = reduce(
            lambda left, right: pd.merge(
                left, right,
                on=['Area'],
                how='inner'
            ),
            table_dfs
        )

        return table_df[YEAR_CODE_COLUMN_NAMES].copy()

In [21]:
table_2022_df = create_table_df_version2('data/medical/2018-2024/CC 2022.xlsx', 2022, False)
table_2023_df = create_table_df_version2('data/medical/2018-2024/CC 2023.xlsx', 2023, True)
table_2024_df = create_table_df_version2('data/medical/2018-2024/CC 2024.xlsx', 2024, False)

# because maguindanao was split to del norte and del sur, we will just combine it again
table_2024_df['Area'] = table_2024_df['Area'].replace({
    'Maguindanao del Norte': 'Maguindanao',
    'Maguindanao del Sur': 'Maguindanao'
})
table_2024_df = (
    table_2024_df
    .groupby('Area', as_index=False)
    .sum(numeric_only=True)
)

table_2024_df['Year'] = 2024

### Cleaning 2025 Data

Changes of the FSIS in their measurements
* 2018 - did not measure IPV
* 2019 - 2021 - measured IPV only once
* 2022 - measured IPV 1 and IPV 2
* 2023 - started using somce the formal names as provided by the PSGC (so Baguio City instead of City of Baguio). However, other names like Manila City was still used
* 2024 - fully used the formal names provided by the PSGC (so City of Marikina) except for some cases like Pasay City (however most likely just a data encoding mistake, this will be fixed with regex)

* 2025 - municipality level data

In [22]:
files = os.listdir('data/medical/2025')
excel_files = sorted([os.path.join('data/medical/2025', file) for file in files if file[0].isdigit()])

# TABLE 1
table1_df = pd.read_excel(
    excel_files[0], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)
table1_df.drop(columns=table1_df.columns[-5], inplace=True)

table1_df.columns = create_column_names(
    ['BCG_early', 'BCG_late', 'HEPA_B1_early', 'HEPA_B1_late', 'CPAB'],
    is_PSGC=True)

# BCG
table1_df['BCG_M'] = table1_df['BCG_early_M'] + table1_df['BCG_late_M']
table1_df['BCG_F'] = table1_df['BCG_early_F'] + table1_df['BCG_late_F']
table1_df['BCG_Total'] = table1_df['BCG_early_Total'] + table1_df['BCG_late_Total']
table1_df['BCG_Percent'] = table1_df['BCG_Total'] / table1_df['Eligible_Pop'] * 100

# HEPA_B1
table1_df['HEPA_B1_M'] = table1_df['HEPA_B1_early_M'] + table1_df['HEPA_B1_late_M']
table1_df['HEPA_B1_F'] = table1_df['HEPA_B1_early_F'] + table1_df['HEPA_B1_late_F']
table1_df['HEPA_B1_Total'] = table1_df['HEPA_B1_early_Total'] + table1_df['HEPA_B1_late_Total']
table1_df['HEPA_B1_Percent'] = table1_df['HEPA_B1_Total'] / table1_df['Eligible_Pop'] * 100
table1_df[['BCG_Percent', 'HEPA_B1_Percent']] = (
    table1_df[['BCG_Percent', 'HEPA_B1_Percent']].fillna(0)
)

relevant_cols = create_column_names(['CPAB', 'BCG', 'HEPA_B1'], is_PSGC=True)
table1_summarized_df = table1_df[relevant_cols].copy()

In [23]:
# TABLE 2
table2_df = pd.read_excel(
    excel_files[1], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table2_df = table2_df.iloc[:, :15]
table2_df.columns = create_column_names(['DPT_1', 'DPT_2', 'DPT_3'], is_PSGC=True)

# TABLE 3
table3_df = pd.read_excel(
    excel_files[2], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table3_df = table3_df.iloc[:, :15]
table3_df.columns = create_column_names(['OPV_1', 'OPV_2', 'OPV_3'], is_PSGC=True)

# TABLE 4
table4_df = pd.read_excel(
    excel_files[3], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table4_df = table4_df.iloc[:, :11]
table4_df.columns = create_column_names(['IPV_1', 'IPV_2'], is_PSGC=True)

# TABLE 5 
table5_df = pd.read_excel(
    excel_files[4], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table5_df = table5_df.iloc[:, :15]
table5_df.columns = create_column_names(['PCV_1', 'PCV_2', 'PCV_3'], is_PSGC=True)

# TABLE 6
table6_df = pd.read_excel(
    excel_files[5], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table6_df = table6_df.iloc[:, :11]
table6_df.columns = create_column_names(['MCV_1', 'MCV_2'], is_PSGC=True)

# TABLE 7 
table7_df = pd.read_excel(
    excel_files[6], sheet_name='Annual',
    skiprows=5,
    skipfooter=4)

table7_df.drop(columns=table7_df.columns[7], inplace=True)
table7_df.columns = create_column_names(['FIC', 'CIC'], is_PSGC=True)

In [25]:
# merge all the 2025 data
table_dfs = [
    table1_df,
    table2_df,
    table3_df,
    table4_df,
    table5_df,
    table6_df,
    table7_df
]

table_dropped_dfs = [table_df.drop(columns=['Eligible_Pop']).copy() for table_df in table_dfs]

for table_dropped_df in table_dropped_dfs:
    table_dropped_df.Area = table_dropped_df.Area.replace(
        {"Maguindanao Sur" : "Maguindanao del Sur",
         "Maguindanao del Sur" : "Maguindanao del Sur"}
    )

# fix the fact that many municipalities can share the same name of Quezon
table_2025_df = reduce(
    lambda left, right: pd.merge(
        left, right.drop(columns=['Area']),
        on='PSGC',
        how='inner'
    ),
    table_dropped_dfs
)

# add Area back once
psgc_area_map = table_dropped_dfs[0][['PSGC', 'Area']].drop_duplicates()
table_2025_df = table_2025_df.drop(columns=['Area'], errors='ignore')
table_2025_df = table_2025_df.merge(psgc_area_map, on='PSGC', how='left')

table_2025_df['Year'] = 2025
table_2025_df = table_2025_df[YEAR_CODE_COLUMN_NAMES + ['PSGC']].copy()

## Combining All of the Data from 2018 to 2025

To construct a consistent panel dataset, different merge keys were used across years due to changes in data granularity and the presence of duplicate area names. For 2018 to 2024, datasets were merged using Area names, as earlier data (especially 2019–2024) was largely at the provincial level, where name duplication across LGUs was not an issue. However, by 2025, the data shifted to a more granular level with many municipalities sharing identical names (e.g., multiple “Quezon”), making PSGC codes necessary as the unique identifier for accurate merging.

Additionally, 2023 was used as the reference year for harmonizing Area names across datasets because it had the smallest and most stable set of entries (138 rows) compared to other years (197 for 2018–2022, 140 for 2024, and 1,744 for 2025). This helped ensure consistency while minimizing mismatches during the merge process.

Some locations (e.g., Negros Occidental, City of Bacolod, Negros Oriental, Siquijor, and Zamboanga Sibugay) had outdated PSGC codes in earlier years; these were manually updated to their latest PSGC equivalents to maintain consistency in the merged dataset.

In [64]:
annual_tables_dfs = [
    table_2018_df,
    table_2019_df,
    table_2020_df,
    table_2021_df,
    table_2022_df,
    table_2023_df,
    table_2024_df,
    table_2025_df
]
annual_tables_dfs = [df.copy() for df in annual_tables_dfs]

# standardized the alternative names
area_map = {
'A.R.M.M.': 'BARMM',
'Autonomous Region in Muslim Mindanao': 'BARMM',
'Compostela Valley': 'Davao de Oro',
'N C R' : 'NCR',
'C A R': 'CAR',
'Philippines':'PHILIPPINES',
'Northern Leyte':'Leyte',
'Mindoro Occidental':'Occidental Mindoro',
'Mindoro Oriental': 'Oriental Mindoro',
'Mt. Province':'Mountain Province',
'Province of Dinagat':'Dinagat Islands',
'Western Samar':'Samar',
'North Cotabato':'Cotabato',
'Maguindanao del Norte': 'Maguindanao',
'Maguindanao del Sur': 'Maguindanao',
'Gen. Santos City':'City of General Santos'
}

def standardize_area(area):
    area = area.strip()

    # standardize the names based on the mapping
    if area in area_map.keys():
        area = area_map[area]
        #print(repr(area))

    # force certain LGUs to be cities
    force_cities= ['Taguig', 'Malabon', 'Navotas', 'Olongapo', 'San Juan']
    if area in force_cities:
        return (f'City of {area}').upper()

    # convert "X City" -> "City of X"
    if area.endswith(' City'):
        name = area.replace(' City', '')
        return (f'City of {name}').upper()

    return area.upper()

In [65]:
# standardize all Area names first
for df in annual_tables_dfs:
    df['Area'] = df['Area'].apply(standardize_area)


# split datasets: up to 2024 vs 2025
tables_pre_2025 = annual_tables_dfs[:-1]
table_2025_df = annual_tables_dfs[-1]


updated_psgc_code = {
 604500000.0: 1804500000,
 630200000.0: 1830200000,
 704600000.0: 1804600000,
 706100000.0: 1806100000,
 908300000.0: 908001000
}

annual_tables_dfs[-3].PSGC = annual_tables_dfs[-3].PSGC.replace(updated_psgc_code)

# use 2023 as the reference for Area to PSGC
psgc_area_map = annual_tables_dfs[-3][['Area', 'PSGC']]

In [76]:
# combine 2018–2024 and align using 2023 Area names
annualized_pre_2025 = pd.concat(tables_pre_2025, ignore_index=True)

annualized_pre_2025 = annualized_pre_2025.merge(
    psgc_area_map,
    on='Area',
    how='right'   # keeps only areas consistent with 2023 #SMTH IS WRONG HEREEEEEEE
)

annualized_pre_2025 = (
    annualized_pre_2025
    .drop(columns='PSGC_x')
    .rename(columns={'PSGC_y':'PSGC'})
    .copy()
)

# handle 2025 using PSGC as the key (more reliable for new splits)
table_2025_cleaned = table_2025_df.merge(
    psgc_area_map,
    on='PSGC',
    how='inner'
)

# if Area columns duplicated, keep the 2023 version
if 'Area_y' in table_2025_cleaned.columns:
    table_2025_cleaned = (
        table_2025_cleaned
        .drop(columns=['Area_x'])
        .rename(columns={'Area_y': 'Area'})
    )


# combine everything into one panel dataset
annualized_completed_df = pd.concat(
    [annualized_pre_2025, table_2025_cleaned],
    ignore_index=True
)

YEAR_PSGC_CODE_COLUMN_NAMES = (
    YEAR_CODE_COLUMN_NAMES[:1] + ['PSGC'] + YEAR_CODE_COLUMN_NAMES[1:]
)

annualized_completed_df = annualized_completed_df[YEAR_PSGC_CODE_COLUMN_NAMES].copy()
annualized_completed_df.sort_values(by=['PSGC', 'Year'], inplace=True)

In [82]:
import pandas as pd

# Columns that should stay single (not grouped)
base_cols = ['Area', 'PSGC', 'Year']


def write_with_grouped_headers(df, writer, sheet_name):
    df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1)
    
    workbook  = writer.book
    worksheet = writer.sheets[sheet_name]
    
    # Formats
    header_main = workbook.add_format({
        'bold': True,
        'align': 'center',
        'valign': 'middle',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })

    header_sub = workbook.add_format({
        'bold': True,
        'align': 'center',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })
        
    highlight_format = workbook.add_format({
        'bg_color': '#FFF2CC'  # light yellow
    })
    
    # ---- STEP 1: Handle base columns ----
    col_idx = 0
    for col in df.columns:
        if col in base_cols:
            worksheet.merge_range(0, col_idx, 1, col_idx, col, header_main)
            col_idx += 1
    
    # ---- STEP 2: Dynamically group remaining columns ----
    grouped_cols = [c for c in df.columns if c not in base_cols]
    
    # Extract prefix (before last "_")
    from collections import defaultdict
    groups = defaultdict(list)
    
    for col in grouped_cols:
        if '_' in col:
            prefix = '_'.join(col.split('_')[:-1])  # e.g. OPV_M → OPV
            groups[prefix].append(col)
        else:
            groups[col].append(col)
    
    # Write grouped headers
    for group, cols in groups.items():
        start_col = col_idx
        end_col = col_idx + len(cols) - 1
        
        # Top header
        worksheet.merge_range(0, start_col, 0, end_col, group, header_main)
        
        # Subheaders
        for i, col in enumerate(cols):
            sub = col.split('_')[-1] if '_' in col else col
            worksheet.write(1, start_col + i, sub, header_sub)
        
        col_idx += len(cols)
    
    # ---- STEP 3: Highlight rows where PSGC ends with 00000000 ----
    if 'PSGC' in df.columns:
        psgc_col_idx = df.columns.get_loc('PSGC')
        
        n_rows = len(df)
        n_cols = len(df.columns)
        
        # Helper to convert column index → Excel letter (handles AA, AB, etc.)
        def colnum_to_excel(n):
            string = ""
            while n >= 0:
                string = chr(n % 26 + 65) + string
                n = n // 26 - 1
            return string
        
        col_letter = colnum_to_excel(psgc_col_idx)
        
        highlight_format = workbook.add_format({
            'bg_color': '#A6A6A6',
            'bold': True
        })
    
    worksheet.conditional_format(
        2, 0,                     # start row, start col
        n_rows + 1, n_cols - 1,   # end row, end col
        {
            'type': 'formula',
            'criteria': f'=RIGHT(TEXT(${col_letter}3,"0"),8)="00000000"',
            'format': highlight_format
        }
    )
    
    # ---- STEP 4: Formatting ----
    worksheet.freeze_panes(2, 0)
    
    for i, col in enumerate(df.columns):
        worksheet.set_column(i, i, 18)


# ---- WRITE FILE ----
with pd.ExcelWriter('outputs/annual_immunization.xlsx', engine='xlsxwriter') as writer:
    
    # All data
    write_with_grouped_headers(annualized_completed_df, writer, 'All_Data')
    
    # Per year
    years = sorted(annualized_completed_df['Year'].unique())
    
    for year in years:
        yearly_df = annualized_completed_df[
            annualized_completed_df['Year'] == year
        ]
        
        write_with_grouped_headers(yearly_df, writer, str(year))